In [ ]:
import sys
import os
import ast
import pandas as pd
import numpy as np # NaN 체크용

# 1. 경로 설정
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.config_loader import load_config

# 2. 데이터 로드
cfg = load_config()
real_data_path = os.path.join(project_root, cfg.path.data.test)
df = pd.read_csv(real_data_path)

# ---------------------------------------------------------
# 🛠️ [핵심] 안전한 파싱 함수 (Fallback Logic)
# ---------------------------------------------------------
def get_problem_data(row):
    """
    1순위: 'question', 'choices' 컬럼에서 가져오기
    2순위: 실패 시 'problems' 컬럼(딕셔너리 스트링) 파싱해서 가져오기
    """
    q = row.get('question')
    c = row.get('choices')

    # 1. 개별 컬럼에 데이터가 잘 들어있는 경우
    if pd.notna(q) and q != "" and pd.notna(c) and c != "":
        # choices가 문자열이면 리스트로 변환
        if isinstance(c, str):
            try:
                c = ast.literal_eval(c)
            except:
                c = []
        return q, c

    # 2. 개별 컬럼이 비어있다면 'problems' 컬럼 파싱 (비상 대책)
    try:
        if pd.notna(row.get('problems')):
            problems_dict = ast.literal_eval(row['problems'])
            return problems_dict.get('question', ''), problems_dict.get('choices', [])
    except Exception as e:
        print(f"⚠️ 파싱 실패: {e}")
    
    return "질문 없음", []

# ---------------------------------------------------------
# 3. State 생성
# ---------------------------------------------------------
row = df.iloc[501]
question, choices = get_problem_data(row)

dummy_state = {
    # --- 입력 데이터 ---
    "paragraph": row.get('paragraph', ''),
    "problem": {
        "question": question,
        "choices": choices
    },

    # --- 처리 데이터 초기화 ---
    "track_info": {},
    "retrieved_context": [],
    "final_prompt_messages": [],
    "solver_results": [],
    "final_answer": None
}

# ---------------------------------------------------------
# 4. 디버깅 출력
# ---------------------------------------------------------
print("\n=== 🔍 데이터 확인 ===")
print(f"1. 컬럼 목록: {df.columns.tolist()}") # 혹시 공백이 있는지 확인용
print(f"2. 원본 problems 데이터: {row.get('problems')}") # 여기가 비어있는지 확인
print("-" * 30)
print("=== ✅ AgentState 구조 확인 ===")
print(f"🔹 지문: {dummy_state['paragraph']}")
print(f"🔹 질문: {dummy_state['problem']['question']}")
print(f"🔹 선택지: {dummy_state['problem']['choices']}")
print(f"🔹 선택지 갯수: {len(dummy_state['problem']['choices'])}")


=== 🔍 데이터 확인 ===
1. 컬럼 목록: ['Unnamed: 0', 'id', 'paragraph', 'problems', 'question_plus']
2. 원본 problems 데이터: {'question': '이 정보가 주어지면?', 'choices': ['한계 소비 성향은 0.80이다.', '한계 저축 성향은 0.20이다.', '한계 저축 성향은 0.10이다.', '한계 저축 성향은 0.90이다.'], 'answer': ''}
------------------------------
=== ✅ AgentState 구조 확인 ===
🔹 지문: 현재 가처분 소득이 10,000달러이고 소비 지출이 8,000달러라고 가정하겠습니다. 가처분 소득이 100달러 증가할 때마다 저축액은 10달러 증가합니다.
🔹 질문: 이 정보가 주어지면?
🔹 선택지: ['한계 소비 성향은 0.80이다.', '한계 저축 성향은 0.20이다.', '한계 저축 성향은 0.10이다.', '한계 저축 성향은 0.90이다.']
🔹 선택지 갯수: 4


In [2]:
from src.agent.nodes.self_querying_retriever import RetrievalNode

# 1. 리트리버 노드 초기화 (모델 로딩)
# cfg는 위에서 이미 로드했으므로 그대로 사용
print("⏳ RetrievalNode 초기화 중...")
retriever = RetrievalNode(cfg)

# 2. 노드 실행 (우리가 만든 dummy_state 주입)
print(f"▶️ 검색 시작: {dummy_state['problem']['question']}")
result = retriever(dummy_state)

# 3. 결과 확인
print("\n" + "="*30)
print(f"🔑 추출된 키워드(로그 확인) -> 검색 결과:")
print("-" * 30)

# 결과가 리스트로 잘 들어왔는지 확인
if result['retrieved_context']:
    print(result['retrieved_context'][0]) 
else:
    print("❌ 검색 결과 없음")
print("="*30)

⏳ RetrievalNode 초기화 중...
🔧 [RetrievalNode] 초기화 중... (사용 모델: main_solver)
🔄 [Loader] 모델 로딩 시작: Qwen3-32B-4bit (unsloth/Qwen3-32B-bnb-4bit)
   ↳ ⚡ 양자화 설정 적용 중...


/data/ephemeral/home/REPO_Jehyeok/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ [Loader] 로딩 완료!
▶️ 검색 시작: 이 정보가 주어지면?
▶️ [Retrieval] 검색 키워드 추출 중...
   - 질문: 이 정보가 주어지면?...
   - 지문 길이: 86자
   ↳ 🔑 추출된 키워드: [한계 저축 성향]

🔑 추출된 키워드(로그 확인) -> 검색 결과:
------------------------------
한계저축성향(限界貯蓄性向)은 소득의 증가에 대한 저축 증가의 비율을 뜻하는 단어이다. 한계소비성향의 경우와 같이 ΔS/ΔY로 나타낸다. 이것은 소득이 변화하는 경우를 말한다.
